In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
from mixing_utils import mix_signals
import numpy as np
import pandas as pd
from tqdm import tqdm

SINR_RANGES = [(-10, 0), (0, 10), (10, 20)]
# SINR_WEIGHTS = [0.50, 0.35, 0.15]
SINR_WEIGHTS = [0.34, 0.33, 0.33]

In [ ]:
def sample_training_sinr(n_samples):
    """
    Generates SINR values based on the weighted distribution.
    """
    # Choose the range bucket for each sample
    range_indices = np.random.choice(
        len(SINR_RANGES), 
        size=n_samples, 
        p=SINR_WEIGHTS
    )
    
    sinrs = []
    for idx in range_indices:
        low, high = SINR_RANGES[idx]
        # Uniformly sample within the chosen bucket
        val = np.random.uniform(low, high)
        sinrs.append(val)
        
    return np.array(sinrs)

In [ ]:
print("Loading Training Tensors...")

# Source A: Augmented Clean Signal (from Stage 04)
clean_path = get_unet_path(STAGE_AUGMENTED, split=TRAIN, signal=CommSignal2)
clean_signals = np.load(clean_path)
print(f"Loaded Clean Target: {clean_signals.shape} from {clean_path}")

# Source B: Interference (from Stage 03 - Split)
interference_map = {
    'Barrage': {'data': np.load(get_unet_path(STAGE_SPLIT, split=TRAIN, signal=BarrageSignal)), 'type': 'Barrage', 'source': 'AWGN_Generator'},
    'Spot':    {'data': np.load(get_unet_path(STAGE_SPLIT, split=TRAIN, signal=EMISignal1)),    'type': 'Spot',    'source': EMISignal1},
    'Digital3':{'data': np.load(get_unet_path(STAGE_SPLIT, split=TRAIN, signal=CommSignal3)),   'type': 'Digital', 'source': CommSignal3},
    'Digital5G':{'data': np.load(get_unet_path(STAGE_SPLIT, split=TRAIN, signal=CommSignal5G1)),'type': 'Digital', 'source': CommSignal5G1}
}
print(f"Loaded Interference Sources: {list(interference_map.keys())}")

In [ ]:
keys = list(interference_map.keys())
n_samples = len(clean_signals)
target_sinrs = sample_training_sinr(n_samples)

mixed_signals = []
metadata_rows = []

print(f"Starting Mixing for {n_samples} samples...")

for i in tqdm(range(n_samples)):
    # Get Clean Sample
    sig_clean = clean_signals[i]
    
    # Randomly Select Interference
    key = np.random.choice(keys)
    selected_int = interference_map[key]

    noise_pool = selected_int['data']

    # randomly select noise sample from the chosen interference type
    noise_idx = np.random.randint(0, len(noise_pool))
    sig_noise = noise_pool[noise_idx]
        
    clean_len = sig_clean.shape[1]
    noise_len = sig_noise.shape[1]

    if noise_len > clean_len:
        # Pick a random start point
        max_start = noise_len - clean_len
        start_idx = np.random.randint(0, max_start)
        sig_noise = sig_noise[:, start_idx : start_idx + clean_len]
    elif noise_len == clean_len:
        sig_noise = sig_noise
    else:
        # Failure (Noise is shorter than signal)
        # We explicitly FORBID looping to prevent artifacts.
        raise ValueError(
            f"CRITICAL DATA ERROR: Interference source '{key}' (ID: {noise_pool}) "
            f"has length {noise_len}, which is shorter than the target {clean_len}. "
            "Looping is forbidden."
        )

    # Get SINR Target
    sinr_db = target_sinrs[i]
    
    # Mix (Physics)
    sig_mixed, alpha = mix_signals(sig_clean, sig_noise, sinr_db)
    
    # Store
    mixed_signals.append(sig_mixed)
    metadata_rows.append({
        'global_index': i,
        'sinr_db': round(sinr_db, 4),
        'clean_frame_id': i,           # ID reference to clean signal
        'noise_frame_id': noise_idx,   # ID reference to noise sample
        'interference_type': selected_int['type'], # e.g., "Spot"
        'source': selected_int['source'],          # e.g., "EMISignal1"
        'alpha': round(alpha, 6)
    })

In [ ]:
output_dir = get_unet_path(STAGE_MIXED, split=TRAIN)
output_npy = output_dir / f"{MIXED_DATASET}.npy"
output_csv = output_dir / f"{MIXED_METADATA}.csv"

np.save(output_npy, np.array(mixed_signals))
pd.DataFrame(metadata_rows).to_csv(output_csv, index=False)

print(f"Saved Mixed Tensor: {output_npy}")
print(f"Saved Metadata Log: {output_csv}")